# Train YOLOV8: Brain Tumor Detection

Self-contained: downloads the Kaggle dataset, builds a simple YOLO dataset from it, trains
yolov8, and saves results + weights to Google Drive. No dependency on any other notebook.

In [ ]:
!pip install -q kagglehub ultralytics pyyaml

In [ ]:
from pathlib import Path
from google.colab import drive
drive.mount('/content/drive')

DRIVE_ROOT = Path('/content/drive/MyDrive/brain-tumor-yolo')
(DRIVE_ROOT / 'weights').mkdir(parents=True, exist_ok=True)
(DRIVE_ROOT / 'results').mkdir(parents=True, exist_ok=True)

import subprocess
print(subprocess.run(['nvidia-smi', '--query-gpu=name', '--format=csv,noheader'],
                      capture_output=True, text=True).stdout.strip())

## Download + build a simple YOLO dataset

In [ ]:
import shutil, kagglehub
import yaml

raw = Path(kagglehub.dataset_download("ahmedsorour1/mri-for-brain-tumor-with-bounding-boxes"))

DATA_DIR = Path('/content/dataset')
shutil.rmtree(DATA_DIR, ignore_errors=True)

def flatten(split, src_root):
    img_out = DATA_DIR / split / 'images'
    lbl_out = DATA_DIR / split / 'labels'
    img_out.mkdir(parents=True, exist_ok=True)
    lbl_out.mkdir(parents=True, exist_ok=True)
    class_id_by_name = {}
    n = 0
    for class_dir in sorted(p for p in src_root.iterdir() if p.is_dir()):
        for img in (class_dir / 'images').glob('*'):
            lbl = class_dir / 'labels' / (img.stem + '.txt')
            if not lbl.exists():
                continue
            dest_name = f"{class_dir.name}_{img.name}"
            shutil.copy2(img, img_out / dest_name)
            shutil.copy2(lbl, lbl_out / (Path(dest_name).stem + '.txt'))
            n += 1
            if class_dir.name not in class_id_by_name:
                first_line = lbl.read_text().splitlines()[0]
                class_id_by_name[class_dir.name] = int(first_line.split()[0])
    print(f"{split}: {n} images")
    return class_id_by_name

train_ids = flatten('train', raw / 'Train')
val_ids = flatten('val', raw / 'Val')

class_id_by_name = {**train_ids, **val_ids}
class_names = [name for name, _ in sorted(class_id_by_name.items(), key=lambda kv: kv[1])]
print("classes:", class_names)

data_yaml = {
    'path': str(DATA_DIR),
    'train': 'train/images',
    'val': 'val/images',
    'nc': len(class_names),
    'names': class_names,
}
with open(DATA_DIR / 'data.yaml', 'w') as f:
    yaml.safe_dump(data_yaml, f, sort_keys=False)
print((DATA_DIR / 'data.yaml').read_text())

## Train

In [ ]:
from ultralytics import YOLO

MODEL_SCALE = 'yolov8s.pt'
IMGSZ = 640
EPOCHS = 50
SEED = 42

last_ckpt = DRIVE_ROOT / 'runs' / 'yolov8' / 'weights' / 'last.pt'
if last_ckpt.exists():
    print(f"Found checkpoint at {last_ckpt}, resuming training from there.")
    model = YOLO(str(last_ckpt))
    model.train(resume=True)
else:
    model = YOLO(MODEL_SCALE)
    model.train(
        data=str(DATA_DIR / 'data.yaml'),
        imgsz=IMGSZ, epochs=EPOCHS, batch=16, seed=SEED, amp=True, patience=15,
        project=str(DRIVE_ROOT / 'runs'), name='yolov8', exist_ok=True,
        save_period=10,
    )

## Training curves (loss, mAP, precision/recall, F1, confusion matrix)

In [ ]:
from PIL import Image
import matplotlib.pyplot as plt

run_dir = Path(model.trainer.save_dir)
curve_files = ['results.png', 'confusion_matrix.png', 'BoxF1_curve.png', 'BoxPR_curve.png', 'BoxP_curve.png', 'BoxR_curve.png']
existing = [run_dir / f for f in curve_files if (run_dir / f).exists()]

if not existing:
    print(f"No curve plots found in {run_dir}; training may not have finished a full validation pass.")
else:
    fig, axes = plt.subplots(1, len(existing), figsize=(7 * len(existing), 6))
    axes = [axes] if len(existing) == 1 else axes
    for ax, path in zip(axes, existing):
        ax.imshow(Image.open(path))
        ax.set_title(path.stem, fontsize=10)
        ax.axis('off')
    plt.tight_layout()
    plt.show()

## Validate

In [ ]:
metrics = model.val(data=str(DATA_DIR / 'data.yaml'))

map50 = float(metrics.box.map50)
map50_95 = float(metrics.box.map)
precision = float(metrics.box.mp)
recall = float(metrics.box.mr)
per_class_ap50 = {model.names[i]: float(ap) for i, ap in enumerate(metrics.box.ap50)}

print(f"mAP50: {map50:.4f}  mAP50-95: {map50_95:.4f}  precision: {precision:.4f}  recall: {recall:.4f}")
print("per-class AP50:", per_class_ap50)

## Save weights + results to Drive

In [ ]:
import json, shutil

best_dst = DRIVE_ROOT / 'weights' / 'yolov8_best.pt'
shutil.copy2(model.trainer.best, best_dst)
print("saved weights:", best_dst)

results_payload = {
    'model_family': 'yolov8',
    'model_scale': MODEL_SCALE,
    'config': {'imgsz': IMGSZ, 'epochs': EPOCHS, 'seed': SEED},
    'metrics': {
        'map50': map50,
        'map50_95': map50_95,
        'precision': precision,
        'recall': recall,
        'per_class_ap50': per_class_ap50,
    },
    'notes': None,
}
results_path = DRIVE_ROOT / 'results' / 'results_yolov8.json'
with open(results_path, 'w') as f:
    json.dump(results_payload, f, indent=2)
print("saved results:", results_path)

## Sample predictions vs. ground truth

In [ ]:
import cv2, glob
import matplotlib.pyplot as plt

sample_imgs = sorted(glob.glob(str(DATA_DIR / 'val' / 'images' / '*')))[:8]
fig, axes = plt.subplots(2, 4, figsize=(20, 10))
for ax, img_path in zip(axes.reshape(-1), sample_imgs):
    pred = model.predict(img_path, imgsz=IMGSZ, verbose=False)[0]
    ax.imshow(cv2.cvtColor(pred.plot(), cv2.COLOR_BGR2RGB))
    ax.set_title(Path(img_path).name, fontsize=8)
    ax.axis('off')
plt.tight_layout()
plt.show()

## Download outputs (weights + results + training run)

In [ ]:
import zipfile
from google.colab import files

zip_path = Path('/content/yolov8_outputs.zip')
run_dir = DRIVE_ROOT / 'runs' / 'yolov8'

with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    zf.write(best_dst, best_dst.name)
    zf.write(results_path, results_path.name)
    for p in run_dir.rglob('*'):
        if p.is_file():
            zf.write(p, f"runs/{p.relative_to(run_dir)}")

print(f"zipped {zip_path} ({zip_path.stat().st_size / 1e6:.1f} MB)")
files.download(str(zip_path))